<a id="unit-u.ae66a49980f54531bf9f751e0069967e"></a>

<div>
  <div>14-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Тақта мен ad hoc ресурстарын бөлу</div>
</div>

11-модуль тақтасы мен ad hoc бір Workload Group бөліседі, ұзақ сканерлеу тақтаны баяулатады. Әрқайсысына топ беріп, орындалу/күту/тыйым/кезекті айналып өтуді көресіз; кезек пен орындалу шегін ажыратасыз. Уақытша қарапайым пайдаланушыны тіркелгі әдепкісімен тақтаға бағыттап, рұқсатсыз топты Doris қабылдамайтынын көресіз. Сүзгісіз сканерлеуді SQL Block Rule арқылы бұғаттап, тәуелділікпен бәрін жойып, қолданбай тұрып ресурс жоспарын тексересіз. Шамамен 45 минут.

**14-модуль: Ресурстарды оқшаулау және сұрауларды қорғау** · [Сабақ](course.md) · [Тест](quiz14_resource_isolation.ipynb) · [Курстың README файлы](../../README.md)

<a id="unit-u.b0c465a1f37a42aaa6b504b08c50822e"></a>

<a id="before-you-begin"></a>

## Бастамас бұрын

Тек сынақ ортасы керек, бұрынғы кестелер қажет емес. Оқу базаңызда `orders_monthly_l3` қайта құрылады.

Workload Group, рөл, пайдаланушы, SQL Block Rule бүкіл кластерге тиесілі. Жұмыс тұрақты `course_dashboard_l3`, `course_adhoc_l3` топтарын, `course_dashboard_reader_l3` рөлін, `course_scan_guard_l3` ережесін; 6-бөлімде кездейсоқ атаулы уақытша пайдаланушыны әдепкі тобымен; жаттығу `course_plan_dashboard_l3`, `course_plan_adhoc_l3` құрады. Әр бөлім өзінікін, 8-бөлім рөл/топтарды, 2-қадам үзілген орындау қалдығын жояды. `normal` мен root өзгермейді.

7-бөлімнің SQL Block Rule-ы барда root-ты қоса барлық сеансқа әсер етеді: ұяшық құрып, тексеріп, `finally` жояды. Үзілсе не ядро қайта басталса, бәрінен бұрын қосылым ұяшығы мен 2-қадамды орындаңыз. Баяу сұраулар `sleep()` қолданады, шын орындалуы үшін SQL кэші өшіріледі. Ортада бір FE, бір BE, CPU cgroup жоқ: CPU шегі орындалмайды.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.access import temporary_user
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect
from dw_course.streaming import wait_for
from dw_course.ui import show_records, show_sql
from dw_course.workload import BackgroundQuery, group_load, run_query, show_outcomes, wait_for_group

lab = connect_sandbox(module=14)

DASHBOARD = "course_dashboard_l3"
ADHOC = "course_adhoc_l3"

<a id="unit-u.6fa4e8165cd14e559b6871d3741f6c36"></a>

<a id="1-which-group-does-a-query-use-now"></a>

## 1. Қазір сұрау қай топты қолданады?

Таңдау реті: сұрауда `SET_VAR(workload_group=...)` hint-і, сеанс `workload_group`, тіркелгі `default_workload_group`, әйтпесе `normal`. Жұмыс hint қолданбайды. Ұяшық root сеанс/әдепкі мәндерін, бар топтарды көрсетіп, `normal` баптауын соңында салыстыруға сақтайды.

In [ ]:
lab.sql("SELECT @@workload_group AS session_workload_group", title="The session's workload_group")
lab.sql("SHOW PROPERTY LIKE 'default_workload_group'", title="The account's default group")
lab.sql("SHOW WORKLOAD GROUPS", title="The Workload Groups that exist",
        columns=["Name", "max_concurrency", "max_queue_size", "queue_timeout", "max_memory_percent", "running_query_num", "waiting_query_num"])
NORMAL_BEFORE = lab.query("SELECT * FROM information_schema.workload_groups WHERE NAME = 'normal'")
expect((lab.query("SELECT @@workload_group"), lab.query("SHOW PROPERTY LIKE 'default_workload_group'")),
       ([("",)], [("default_workload_group", "normal")]), title="Root's queries run in normal")

<a id="unit-u.8d897618d9ac4c489c804fc99c783041"></a>

**Күтілетін нәтиже**

Сеанс `workload_group` бос, әдепкі `normal`, `normal` тобы — `max_concurrency` 2147483647, `max_queue_size` 0, `queue_timeout` 0, `max_memory_percent` 100%: іс жүзінде шексіз, кезексіз. Жасыл: root `normal` ішінде. Курс тобы да көрінсе, бұрынғы орындау үзілген; 2-қадам жояды.

<a id="unit-u.7180be970b5c4fbfbc2cff83703c7780"></a>

<a id="2-prepare-the-table-and-remove-leftovers"></a>

## 2. Кестені дайындап, қалдықтарды жою

Алдыңғы кластерлік нысандарды жою реті керек: әдепкісі курс тобы болған пайдаланушы топты ұстайды; жаһандық ереже төмендегі толық сканерлеуді бұғаттар еді. Ұяшық 6-бөлім қалдық пайдаланушыларын, кейін ереже/рөл/топтарды жойып, `orders_monthly_l3` кестесін 13-модуль айлық бөлімдерімен, айына бір тапсырыспен қайта құрады.

In [ ]:
import pymysql

with lab.connection.cursor(pymysql.cursors.DictCursor) as cursor:
    cursor.execute("SHOW ALL GRANTS")
    leftover_users = [row["UserIdentity"] for row in cursor.fetchall() if row["UserIdentity"].startswith("'course_routing_l3_")]
for identity in leftover_users:
    lab.execute(f"DROP USER IF EXISTS {identity}")
lab.execute("DROP SQL_BLOCK_RULE IF EXISTS course_scan_guard_l3")
lab.execute("DROP ROLE IF EXISTS course_dashboard_reader_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_dashboard_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_adhoc_l3")

lab.execute("DROP TABLE IF EXISTS orders_monthly_l3")
lab.execute("""
CREATE TABLE orders_monthly_l3 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    amount DECIMAL(18,2) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p202501 VALUES LESS THAN ("2025-02-01"),
    PARTITION p202502 VALUES LESS THAN ("2025-03-01"),
    PARTITION p202503 VALUES LESS THAN ("2025-04-01")
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("orders_monthly_l3", ["order_date", "order_id", "amount"], [("2025-01-15", 140001, 100), ("2025-02-15", 140002, 200), ("2025-03-15", 140003, 300)])
lab.sql("SELECT * FROM orders_monthly_l3 ORDER BY order_date, order_id", title="The monthly table")
expect(lab.query("SELECT CAST(order_date AS STRING), order_id, amount FROM orders_monthly_l3 ORDER BY order_date, order_id"),
       [("2025-01-15", 140001, "100.00"), ("2025-02-15", 140002, "200.00"), ("2025-03-15", 140003, "300.00")],
       title="One order in each of three months")

<a id="unit-u.5f3c0db2858f4c15a7bb9658f84c7edd"></a>

**Күтілетін нәтиже**

Үш тапсырыс: 2025-01-15 күнгі 140001 (100.00), 2025-02-15 күнгі 140002 (200.00), 2025-03-15 күнгі 140003 (300.00); әр бөлімге бір, жасыл тексеру. 4-бөлім баяуы тек наурызды, 7-бөлім сүзгісізі үшеуін оқиды.

<a id="unit-u.7e7ff141b3664c70baddd3134f722b46"></a>

<a id="3-create-the-dashboard-and-ad-hoc-groups"></a>

## 3. Тақта және ad hoc топтарын құру

Баптаулар 14.6 нұсқаулығынан. Ad hoc қатар орындалуы/кезегі 1: аз сұраумен күту/тыйым көрінеді.

BE-де CPU cgroup бапталмаған, max_cpu_percent орындалмайды. Курс рөліне тек тақта тобының USAGE_PRIV рұқсатын беріңіз.

In [ ]:
lab.execute("""
CREATE WORKLOAD GROUP course_dashboard_l3
PROPERTIES (
    "max_concurrency" = "8",
    "max_queue_size" = "20",
    "queue_timeout" = "10000",
    "max_memory_percent" = "50"
)
""")
lab.execute("""
CREATE WORKLOAD GROUP course_adhoc_l3
PROPERTIES (
    "max_concurrency" = "1",
    "max_queue_size" = "1",
    "queue_timeout" = "30000",
    "max_memory_percent" = "30",
    "max_cpu_percent" = "30"
)
""")
lab.execute("CREATE ROLE course_dashboard_reader_l3")
lab.execute("GRANT USAGE_PRIV ON WORKLOAD GROUP 'course_dashboard_l3' TO ROLE 'course_dashboard_reader_l3'")
lab.sql("SHOW WORKLOAD GROUPS LIKE 'course%'", title="The two course Workload Groups",
        columns=["Name", "max_concurrency", "max_queue_size", "queue_timeout", "max_memory_percent", "max_cpu_percent"])
expect(
    lab.query("""
SELECT NAME, MAX_CONCURRENCY, MAX_QUEUE_SIZE, QUEUE_TIMEOUT, MAX_MEMORY_PERCENT
FROM information_schema.workload_groups
WHERE NAME IN ('course_dashboard_l3', 'course_adhoc_l3')
ORDER BY NAME
"""),
    [
        ("course_adhoc_l3", 1, 1, 30000, "30%"),
        ("course_dashboard_l3", 8, 20, 10000, "50%"),
    ],
    title="Limits for both groups",
)
expect(
    lab.query("""
SELECT GRANTEE, WORKLOAD_GROUP_NAME, PRIVILEGE_TYPE
FROM information_schema.workload_group_privileges
WHERE WORKLOAD_GROUP_NAME IN ('course_dashboard_l3', 'course_adhoc_l3')
ORDER BY GRANTEE, WORKLOAD_GROUP_NAME
"""),
    [("course_dashboard_reader_l3", "course_dashboard_l3", "Usage_priv")],
    title="Course role has usage only on the dashboard group",
)

<a id="unit-u.cd0f9d9f15464bfe9e9b22d0b1a4b584"></a>

**Күтілетін нәтиже**

Екі топ: `course_adhoc_l3` — 1 қатар орындалу, 1 кезек, 30000 ms, 30% жады, 30% CPU; `course_dashboard_l3` — 8, 20, 10000 ms, 50% жады, 100% CPU. Екі жасыл: шектер, рөлдің тек тақта тобындағы `Usage_priv` рұқсаты.

`USAGE_PRIV` қолдануға мүмкіндік береді, бағыттамайды. 6-бөлім тіркелгі әдепкісімен бағыттайды. CPU cgroup жоқ, 30% — мұнда орындатылмайтын баптау.

<a id="unit-u.855d5c52a8d64a499b51f1b00be4777d"></a>

<a id="4-submit-five-queries-run-queue-or-reject"></a>

## 4. Бес сұрау: орындала ма, күте ме, қабылданбай ма?

sleep() наурыздың жалғыз жолын баяу сканерлеуді имитациялайды, sleep(6) шамамен алты секунд. Әр сұрау жеке сеанс ашып, SQL Cache өшіреді, айнымалыларын орнатып, орындалады.

Кэш бұрынғы нәтижені кезексіз қайтарар еді, сондықтан осы тәжірибеде өшіру қажет.

- A ad hoc жалғыз орнын алады.
- B артынан кезекке тұрады.
- C кезек толғанда сол топқа келіп, қабылданбауы керек.
- D тақтаның өз орнын қолданады.
- E ad hoc ішінде bypass_workload_group қосады.

Уақыт жарысын болдырмауға тәуелді әр сұраудан бұрын күтілетін орындалған/күткен сандарды күтіңіз.

In [ ]:
SLOW = "SELECT order_id, sleep({seconds}) AS slow_scan FROM orders_monthly_l3 WHERE order_date >= '2025-03-01'"

a = BackgroundQuery(lab, "A ad hoc", SLOW.format(seconds=6), workload_group=ADHOC)
wait_for_group(lab, ADHOC, running=1, waiting=0)
b = BackgroundQuery(lab, "B ad hoc", SLOW.format(seconds=1), workload_group=ADHOC)
wait_for_group(lab, ADHOC, running=1, waiting=1)

lab.sql("""
SELECT q.QUERY_STATUS, g.NAME AS workload_group, q.QUEUE_START_TIME, q.QUEUE_END_TIME
FROM information_schema.active_queries q
JOIN information_schema.workload_groups g ON q.WORKLOAD_GROUP_ID = g.ID
WHERE g.NAME = 'course_adhoc_l3'
ORDER BY q.QUERY_STATUS
""", title="A is running; B is queued");

<a id="unit-u.0f76d2bfeee8484e93f13142e1313624"></a>

**Күтілетін нәтиже**

Ad hoc үшін `active_queries` екі жолы: A `RUNNING`, екі кезек уақыты орнатылған; B `WAIT_IN_QUEUE`, `QUEUE_START_TIME` бар, `QUEUE_END_TIME` жоқ — әлі орын алмады.

In [ ]:
c = run_query(lab, "C ad hoc", SLOW.format(seconds=1), workload_group=ADHOC)
d = run_query(lab, "D dashboard", SLOW.format(seconds=1), workload_group=DASHBOARD)
e = run_query(lab, "E queue bypass", SLOW.format(seconds=1), workload_group=ADHOC, bypass_workload_group=True)
load_while_a_runs = group_load(lab, ADHOC)

a_outcome, b_outcome = a.result(), b.result()
show_outcomes("Outcomes of five queries", [a_outcome, b_outcome, c, d, e])
wait_for_group(lab, ADHOC, running=0, waiting=0)
expect(
    [(o.label, o.result) for o in (a_outcome, b_outcome, c, d, e)],
    [
        ("A ad hoc", "OK"),
        ("B ad hoc", "OK"),
        ("C ad hoc", "query waiting queue is full, queue capacity=1, waiting num=1"),
        ("D dashboard", "OK"),
        ("E queue bypass", "OK"),
    ],
    title="Execution, queuing, rejection, and bypass",
)
expect(load_while_a_runs, {"running": 1, "waiting": 1}, title="Ad hoc counters after C, D, and E finish")
expect(b_outcome.seconds > d.seconds + 3, True, title="B elapsed time includes queue wait")

<a id="unit-u.92ffd2bcc3a7438b938c626ba1ef57ab"></a>

**Күтілетін нәтиже**

Бес нәтиже: A шамамен 6 секундта OK; B A-ны күтіп 7-де OK; C бірден `query waiting queue is full, queue capacity=1, waiting num=1`; тақтадағы D шамамен 1-де OK; кезексіз E шамамен 1-де OK. Үш жасыл: бес нәтиже, C/D/E кезінде бір орындалған/бір күткен ad hoc саны, B уақыты күтуді қамтиды.

B кезектің жалғыз орнын алғандықтан C қабылданбады. D күтпеді, өз орны бар. E C-мен бір топта әр пайдаланушы қоя алатын `bypass_workload_group` арқылы өтті: кезек оқшаулау шекарасы емес, пайдалану рұқсаты мен BE жады шегі әлі қолданылады.

<a id="unit-u.6da8f253a1724fe3a1d2b1a0b55acc76"></a>

<a id="5-limit-the-queue-wait-and-the-execution-separately"></a>

## 5. Кезек күтуін және орындалуды бөлек шектеу

queue_timeout күтуді, query_timeout орындалуды шектейді.

Алдымен ad hoc күтуін бір секундқа қысқартып, бос емес орын сынағын қайталаңыз. Кейін төрт секундтық F сұрауын екі секундтық орындалу шегімен жіберіңіз. F топ таңдамайды, normal қолданады.

Қадам күту шегін өзгертеді. 3-қадамды бастапқы баптаумен қайталауға 1-қадамнан бастаңыз.

In [ ]:
lab.execute('ALTER WORKLOAD GROUP course_adhoc_l3 PROPERTIES ("queue_timeout" = "1000")')

a = BackgroundQuery(lab, "A ad hoc", SLOW.format(seconds=3), workload_group=ADHOC)
wait_for_group(lab, ADHOC, running=1, waiting=0)
b = run_query(lab, "B queue timeout", SLOW.format(seconds=1), workload_group=ADHOC)
f = run_query(lab, "F execution timeout", SLOW.format(seconds=4), query_timeout=2)

a_outcome = a.result()
show_outcomes("Queue and execution timeouts", [a_outcome, b, f])
wait_for_group(lab, ADHOC, running=0, waiting=0)
expect(
    [(o.label, o.result) for o in (a_outcome, b, f)],
    [
        ("A ad hoc", "OK"),
        ("B queue timeout", "query queue timeout, timeout: 1000 ms"),
        ("F execution timeout", "query timeout"),
    ],
    title="Two distinct timeout errors",
)
expect(b.seconds < 2, True, title="B stops waiting after about one second")
expect(1.5 <= f.seconds < 4, True, title="F is cancelled after about two seconds of execution")

<a id="unit-u.63d8f2a576c9433d9c8d01bd761ca7ab"></a>

**Күтілетін нәтиже**

A шамамен 3 секундта OK; B 1 секундта `query queue timeout, timeout: 1000 ms`; `normal` ішіндегі F 2 секундта `query timeout`. Үш жасыл: екі қате, B күтуі 2 секундтан аз, F — 1.5–4 секунд.

B орын алмаған, BE-де орындалмаған. F тоқтатылғанша ресурс қолданды. Пайдаланушы сеанс `query_timeout` көтере алады, әкімші орнатқан сол тіркелгі қасиеті басым, пайдаланушы өзгерте алмайды.

<a id="unit-u.516dc4590ebf4a4c8a7a312109e12a87"></a>

<a id="6-route-an-ordinary-user-with-its-accounts-default"></a>

## 6. Қарапайым пайдаланушыны тіркелгі әдепкісімен бағыттау

Әзірге бәрін root орындады. Тақта тіркелгісі сеанстарында баптамай өз тобына түсуі керек: рөліне `USAGE_PRIV`, тіркелгісіне `default_workload_group` беріңіз. Ұяшық рөлге кестені оқу, уақытша пайдаланушыға рөл беріп, әкімшінің `SET PROPERTY FOR` командасымен әдепкісін орнатады.

Сол пайдаланушымен баяу сұрау орындап, `active_queries` ішінен тобын оқиды. Кейін рөлге рұқсатсыз ad hoc тобын сеансқа қояды; жауабын болжаңыз. `finally` ішінде әдепкісін `normal` етіп, пайдаланушыны жояды: біреудің әдепкі тобы жойылмайды.

In [ ]:
lab.execute(f"GRANT SELECT_PRIV ON internal.{lab.database}.orders_monthly_l3 TO ROLE 'course_dashboard_reader_l3'")
SLOW = "SELECT order_id, sleep({seconds}) AS slow_scan FROM orders_monthly_l3 WHERE order_date >= '2025-03-01'"
with temporary_user(lab, "course_dashboard_reader_l3", prefix="course_routing_l3") as user:
    default_sql = f"SET PROPERTY FOR '{user.name}' 'default_workload_group' = '{DASHBOARD}'"
    show_sql("Route the user's queries to the dashboard group", default_sql + ";")
    lab.execute(default_sql)
    try:
        g = BackgroundQuery(lab, "G as the dashboard user", SLOW.format(seconds=3), user=user)
        running = wait_for(lambda: lab.query("""SELECT g.NAME, q.QUERY_STATUS FROM information_schema.active_queries q
            JOIN information_schema.workload_groups g ON q.WORKLOAD_GROUP_ID = g.ID WHERE q.USER = %s""", (user.name,)),
            bool, description="Find the user's running query", timeout=10)
        h = run_query(lab, "H as the dashboard user, in the ad hoc group", SLOW.format(seconds=1), user=user, workload_group=ADHOC)
        show_outcomes("The dashboard user's two queries", [g.result(), h])
        show_records("Where the user's query ran", [{"user": user.name, "workload_group": running[0][0], "status": running[0][1]}])
        expect(running[0][0], DASHBOARD, title="The user's query runs in its default group")
        expect(h.result.startswith(f"Access denied; you need (at least one of) the USAGE/ADMIN privilege(s) to use workload group '{ADHOC}'"),
               True, title="The user may not use the ad hoc group")
    finally:
        lab.execute(f"SET PROPERTY FOR '{user.name}' 'default_workload_group' = 'normal'")

<a id="unit-u.0653ec19d65d47be9733257e2c235570"></a>

**Күтілетін нәтиже**

Пайдаланушыны құрып, әдепкісін орнату; кейін G шамамен 3 секундта OK, H бірден `Access denied; you need (at least one of) the USAGE/ADMIN privilege(s) to use workload group 'course_adhoc_l3'. User: 'course_routing_l3_<random>'@'%'` алады. Сұрауы `course_dashboard_l3` ішінде `RUNNING`. Екі жасыл тексеру.

Пайдаланушы топ атамады: әдепкісі бағыттады. Рұқсат айнымалыны орнатқанда емес, орындалуда тексеріледі: `SET workload_group` өтті, сұрау өтпеді. `USAGE_PRIV` жоқ әдепкіде әр сұрау солай сәтсіз болар еді, екеуін бірге орнатыңыз.

<a id="unit-u.947c85c547184daea8a952e67a7edaf1"></a>

<a id="7-reject-a-full-scan-with-a-sql-block-rule"></a>

## 7. SQL Block Rule арқылы толық сканерлеуді қабылдамау

Global ереже ең көбі екі бөлімге рұқсат етеді. Шектеусіз сұрау үшті оқып, өтпеуі тиіс; тек наурыз біреуге қысқарып өтеді.

Ереже root қоса барлық сеансқа қатысты. Бір ұяшықта try/finally арқылы құрып, тексеріп жойыңыз. Үзілу/ядроны қайта қосу қалдыруы мүмкін; 2-қадам жояды.

SUM(amount) қолданыңыз: сүзгісіз COUNT(*) 14.5-бөлімдегідей бөлім оқымай жауап ала алады.

In [ ]:
lab.execute("""
CREATE SQL_BLOCK_RULE course_scan_guard_l3
PROPERTIES ("partition_num" = "2", "global" = "true", "enable" = "true")
""")
try:
    lab.sql("SHOW SQL_BLOCK_RULE FOR course_scan_guard_l3", title="Scan rule")
    full_scan = run_query(lab, "Full-table SUM", "SELECT SUM(amount) FROM orders_monthly_l3")
    one_month = run_query(lab, "March only", "SELECT SUM(amount) FROM orders_monthly_l3 WHERE order_date >= '2025-03-01'")
finally:
    lab.execute("DROP SQL_BLOCK_RULE IF EXISTS course_scan_guard_l3")

show_outcomes("Scan-rule outcomes", [full_scan, one_month])
expect(
    [(o.label, o.result) for o in (full_scan, one_month)],
    [
        ("Full-table SUM", "sql hits sql block rule: course_scan_guard_l3, reach partition_num : 2"),
        ("March only", "OK"),
    ],
    title="Scan rule rejects the full scan",
)
expect(
    [row for row in lab.query("SHOW SQL_BLOCK_RULE") if row[0] == "course_scan_guard_l3"],
    [],
    title="Global rule has been removed",
)

<a id="unit-u.5fd4449ed6174db6931db2c89913d60a"></a>

**Күтілетін нәтиже**

Ереже: `PartitionNum` 2, `Global` true, `Enable` true; толық кесте `SUM` — `sql hits sql block rule: course_scan_guard_l3, reach partition_num : 2`, наурыз — OK. Екі жасыл: толық сканерлеу қабылданбады, ереже жойылды.

Сұрау мүлде орындалмады. 2 — шек, қажет болған 3 бөлім емес; наурыз біреуін оқыды. Ереже секундтан аз тұрса да, сол кезде барлық сеансқа қатысты. Тек аналитиктерге керек болса, әкімшінің `sql_block_rules` қасиетімен тіркелгілеріне байлаңыз.

<a id="unit-u.8b03db4bf64a49a49e55ce1d03cf9784"></a>

<a id="8-revoke-access-and-remove-the-role-and-the-groups"></a>

## 8. Рұқсатты қайтарып, рөл/топтарды жою

Алдымен тәуелділік: еш Workload Policy сілтемейді, 6-бөлім пайдаланушы әдепкісін `normal` етіп, пайдаланушыны жойды.

Кейін `USAGE_PRIV` қайтарып, рөлді, екі топты жойыңыз: рұқсат топтан ұзақ сақталып, сол атпен қайта құрылғанына қайтады. orders_monthly_l3 тексеруге қалады; келесіде 2-қадам қайта құрады.

In [ ]:
expect(
    lab.query("""
SELECT COUNT(*) FROM information_schema.workload_policy
WHERE WORKLOAD_GROUP IN ('course_dashboard_l3', 'course_adhoc_l3')
"""),
    [(0,)],
    title="No Workload Policy references the course groups",
)
lab.execute("REVOKE USAGE_PRIV ON WORKLOAD GROUP 'course_dashboard_l3' FROM ROLE 'course_dashboard_reader_l3'")
lab.execute("DROP ROLE IF EXISTS course_dashboard_reader_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_dashboard_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_adhoc_l3")
lab.sql("SHOW WORKLOAD GROUPS", title="The Workload Groups after cleanup",
        columns=["Name", "max_concurrency", "max_queue_size", "queue_timeout", "max_memory_percent"])
expect(
    lab.query("""
SELECT
    (SELECT COUNT(*) FROM information_schema.workload_groups
     WHERE NAME IN ('course_dashboard_l3', 'course_adhoc_l3')),
    (SELECT COUNT(*) FROM information_schema.workload_group_privileges
     WHERE WORKLOAD_GROUP_NAME IN ('course_dashboard_l3', 'course_adhoc_l3'))
"""),
    [(0, 0)],
    title="Course groups and usage grants have been removed",
)
expect([row[0] for row in lab.query("SHOW ROLES") if row[0] == "course_dashboard_reader_l3"], [], title="Course role has been removed")
expect(lab.query("SELECT * FROM information_schema.workload_groups WHERE NAME = 'normal'"), NORMAL_BEFORE, title="normal remains unchanged")

<a id="unit-u.f2b9f9d44fa644a79da6bcec3b3bb266"></a>

**Күтілетін нәтиже**

Жасыл тексеру: саясат сілтемейді; қалған топ — тек `normal`; кейін үш жасыл: курс топтары/рұқсаттары, рөл жойылды, `normal` 1-қадам баптауында. `orders_monthly_l3` сұрауға қалады; келесі орындауда 2-қадам қайта құрады.

<a id="unit-u.0d6efbf7b2564faa8a233c8c7c1a8770"></a>

<a id="9-practice-check-a-resource-plan-before-you-apply-it"></a>

## 9. Жаттығу: ресурс жоспарын қолданбай тұрып тексеру

Тақта үш FE кластерінде басталады. Бес BI тіркелгісі жаңарғанда шамамен 20 сұрауды қатар, үш аналитик тұрақсыз сканерлеулер жібереді. Жоспарды сөздікпен жазыңыз: топтың қатар орындалуы, кезегі, миллисекунд күтуі. Алдымен Python-да тексеріңіз: біркелкі қолданылған үш FE тақтасының орындары 20 сұрауды ұстасын, ad hoc кезегі шағын қалсын. Кейін `course_plan_dashboard_l3`, `course_plan_adhoc_l3` сынақ атауларымен топ құрып, `information_schema.workload_groups` арқылы баптауын қайта оқып салыстырып, жойыңыз.

In [ ]:
# Your code here

<a id="unit-u.42dfbb11241c48d8a6aedbe94a237c6a"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған. Салыстыру үшін ашыңыз немесе орындаңыз.

In [ ]:
plan = {
    "course_plan_dashboard_l3": {"max_concurrency": 8, "max_queue_size": 20, "queue_timeout": 10000},
    "course_plan_adhoc_l3": {"max_concurrency": 1, "max_queue_size": 2, "queue_timeout": 30000},
}
FES, PEAK = 3, 20
expect((plan["course_plan_dashboard_l3"]["max_concurrency"] * FES >= PEAK, plan["course_plan_adhoc_l3"]["max_queue_size"] <= 5),
       (True, True), title="The plan holds the peak on three FEs and keeps the ad hoc queue small")
try:
    for name, settings in plan.items():
        properties = ", ".join(f'"{key}" = "{value}"' for key, value in settings.items())
        lab.execute(f"DROP WORKLOAD GROUP IF EXISTS {name}")
        lab.execute(f"CREATE WORKLOAD GROUP {name} PROPERTIES ({properties})")
    applied = lab.sql("""SELECT NAME, MAX_CONCURRENCY, MAX_QUEUE_SIZE, QUEUE_TIMEOUT FROM information_schema.workload_groups
                         WHERE NAME LIKE 'course_plan_%' ORDER BY NAME""", title="The groups as Doris applied them")
    expect([tuple(row) for row in applied.itertuples(index=False)],
           sorted((name, s["max_concurrency"], s["max_queue_size"], s["queue_timeout"]) for name, s in plan.items()),
           title="Doris applied the plan as written")
finally:
    for name in plan:
        lab.execute(f"DROP WORKLOAD GROUP IF EXISTS {name}")

<a id="unit-u.6a59c97a5f924defbf04bb19b1108c2f"></a>

**Күтілетін нәтиже**

Жасыл: үш FE × 8 = 24 орын 20 шың сұрауға жетеді, ad hoc кезегі 2. Қолданылғаны: `course_plan_adhoc_l3` — 1, 2, 30000; `course_plan_dashboard_l3` — 8, 20, 10000; жасыл тексеру, кейін екеуі жойылады.

Кезек FE-ге жеке: 24 тек қосылымдар таралса бар; бәрі бір FE-ге барса, тақтада 8 орын/20 кезек. Жиынтыққа сенбес бұрын таралуды тексеріп, 6-бөлімдегідей `USAGE_PRIV`, `default_workload_group` арқылы бағыттаңыз.

<a id="unit-u.e63b5ac4e63e4734a88c00344d659fbc"></a>

<a id="you-are-done-when"></a>

## Аяқтау шарттары

- таңдау ретін, root/тақта пайдаланушысы қай топта болғанын білесіз;
- қатар орындалу/кезек/күту бойынша нәтижені болжап, `bypass_workload_group` неге оқшаулау шекарасы еместігін айта аласыз;
- `query queue timeout`, `query timeout` ажыратып, қайсысы BE ресурсын қолданғанын білесіз;
- пайдаланушы әдепкісінде орындалып, ad hoc-қа тыйым алды, `USAGE_PRIV`, `default_workload_group` әрқайсысын түсіндіресіз;
- SQL Block Rule толық сканерлеуді басталмай қабылдамады, ауқымын білесіз;
- тәуелділікпен бәрін жойдыңыз, `normal` өзгермеді.

<a id="unit-u.34b4b8ad38e44d64822b00ad29216e42"></a>

<a id="next"></a>

## Әрі қарай

[14-тест: Ресурстарды оқшаулау және сұрауларды қорғау](quiz14_resource_isolation.ipynb) файлын ашыңыз. Бұл — соңғы жұмыс; [курстың README файлы](../../README.md) құрғаныңызды жинақтайды. [Тазалау](../../README.md#clean-up) арқылы деректермен кідіртіп/жалғастырыңыз не қосымшалардан кейін барлық ресурсты берілген тәуелділік ретімен жойыңыз.